In [1]:
import torch
import torch.nn as nn
import math
import os
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List
# !pip install pytorch_lightning

In [2]:
src_path = Path('.').absolute().parent
data_path = src_path / '/content/drive/MyDrive/Colab Notebooks/google-colab/data/kdd17'

In [3]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Tuple, Dict

class GARCHNetDataset(Dataset):
    def __init__(self, data_path: str, window_size: int, dataset_type: str = 'kdd17', split: str = 'train'):
        """
        Custom dataset for GARCHNet time series prediction

        Args:
            data_path: Path to data directory
            window_size: Size of sliding window for sequence creation
            dataset_type: Type of dataset ('kdd17' or 'acl18')
            split: Data split to use ('train', 'val', or 'test')
        """
        self.window_size = window_size
        self.split = split

        # Dataset configuration
        self.ds_config = {
            'kdd17': {
                'path': '/content/drive/MyDrive/Colab Notebooks/google-colab/data/kdd17/price_long_50',
                'date_path': '/content/drive/MyDrive/Colab Notebooks/google-colab/data/kdd17/trading_dates.csv',
                'train_date': '2015-01-01',
                'val_date': '2016-01-01',
                'test_date': '2017-01-01'
            },
            'acl18': {
                'path': 'stocknet-dataset/price/raw',
                'date_path': 'stocknet-dataset/price/trading_dates.csv',
                'train_date': '2015-08-01',
                'val_date': '2015-10-01',
                'test_date': '2016-01-01'
            }
        }[dataset_type]

        self.data_dir = Path(data_path) / self.ds_config['path']
        if not self.data_dir.exists():
            raise FileNotFoundError(f"Data directory not found: {self.data_dir}")

        # Load and process data
        self.processed_data = self.load_dataset()
        self.sequences: List[np.ndarray] = []
        self.targets: List[np.ndarray] = []
        self.prepare_split_sequences()

    def prepare_split_sequences(self) -> None:
        """Prepare sequences for the specified split"""
        for df in self.processed_data:
            train_idx, val_idx, test_idx = self.split_data(df)

            split_mask = {
                'train': train_idx,
                'val': val_idx,
                'test': test_idx
            }[self.split]

            split_data = df[split_mask].reset_index(drop=True)

            if len(split_data) > self.window_size:
                seq, targ = self.create_sequences(split_data)
                self.sequences.extend(seq)
                self.targets.extend(targ)

    @staticmethod
    def calculate_features(df: pd.DataFrame) -> pd.DataFrame:
        """Calculate technical indicators and features"""
        # Price ratios
        price_ratios = (df.loc[:, ['Open', 'High', 'Low']].div(df['Close'], axis=0) - 1) * 100
        price_ratios.columns = ['open', 'high', 'low']

        # Returns
        returns = pd.DataFrame({
            'close': df['Close'].pct_change() * 100,
            'adj_close': df['Adj Close'].pct_change() * 100
        })

        # Volume features
        volume_features = pd.DataFrame({
            'volume_change': df['Volume'].pct_change() * 100,
            'log_volume': np.log(df['Volume'] + 1)
        })

        # Moving averages
        ma_features = pd.DataFrame()
        for k in [5, 10, 20]:
            # Volume trend
            ma_features[f'volume_ma{k}'] = df['Volume'].rolling(k).mean() / df['Volume']
            # Volume volatility
            ma_features[f'volume_std{k}'] = df['Volume'].rolling(k).std() / df['Volume']

        # Price trends
        for k in [5, 10, 15, 20, 25, 30]:
            ma_features[f'price_trend{k}'] = ((df['Adj Close'].rolling(k).sum() /
                                             (k * df['Adj Close'])) - 1) * 100

        return pd.concat([
            df[['Date']].rename(columns={'Date': 'date'}),
            price_ratios,
            returns,
            volume_features,
            ma_features
        ], axis=1)

    def load_single_tick(self, file_path: Path) -> pd.DataFrame:
        """Process single ticker data"""
        df = pd.read_csv(file_path)
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.sort_values('Date').reset_index(drop=True)

        # Handle column naming
        if 'Unnamed: 7' in df.columns:
            df = df.drop(columns='Unnamed: 7')
        if 'Original_Open' in df.columns:
            df = df.rename(columns={'Original_Open': 'Open', 'Open': 'Adj Open'})

        # Calculate features
        df_processed = self.calculate_features(df)

        # Remove NaN values
        max_nan_col = df_processed.columns[df_processed.isnull().sum() == df_processed.isnull().sum().max()]
        return df_processed.loc[~df_processed[max_nan_col].isnull().values, :]

    def load_dataset(self) -> List[pd.DataFrame]:
        """Load and process all ticker data"""
        tick_files = [p for p in self.data_dir.glob('*') if p.is_file()]
        return [self.load_single_tick(file_path) for file_path in tick_files]

    def create_sequences(self, data: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        """Create sequences for time series prediction"""
        feature_cols = [col for col in data.columns if col != 'date']
        values = data[feature_cols].values

        sequences = []
        targets = []

        for i in range(len(values) - self.window_size):
            sequences.append(values[i:(i + self.window_size)])
            targets.append(values[i + self.window_size])

        return np.array(sequences), np.array(targets)

    def split_data(self, data: pd.DataFrame) -> Tuple[pd.Series, pd.Series, pd.Series]:
        """Split data into train/val/test sets based on dates"""
        return (
            data['date'] < self.ds_config['train_date'],
            (self.ds_config['train_date'] <= data['date']) & (data['date'] < self.ds_config['val_date']),
            (self.ds_config['val_date'] <= data['date']) & (data['date'] < self.ds_config['test_date'])
        )

    def __len__(self) -> int:
        """Return total number of sequences"""
        return len(self.sequences)

    def __getitem__(self, idx: int) -> Tuple[torch.FloatTensor, torch.FloatTensor]:
        """Get a single sequence-target pair"""
        return torch.FloatTensor(self.sequences[idx]), torch.FloatTensor(self.targets[idx])

def create_dataloaders(
    data_path: str,
    window_size: int,
    batch_size: int = 32,
    dataset_type: str = 'kdd17',
    num_workers: int = 0
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """
    Create train, validation, and test data loaders

    Args:
        data_path: Path to data directory
        window_size: Size of sliding window for sequence creation
        batch_size: Batch size for DataLoader
        dataset_type: Type of dataset ('kdd17' or 'acl18')
        num_workers: Number of worker processes for data loading

    Returns:
        Tuple of (train_loader, val_loader, test_loader)
    """
    datasets = {
        split: GARCHNetDataset(data_path, window_size, dataset_type, split)
        for split in ['train', 'val', 'test']
    }

    for split, dataset in datasets.items():
        print(f"{split.capitalize()} sequences: {len(dataset)}")

    loader_args = dict(
        batch_size=batch_size,
        shuffle=False,  # No shuffling for time series
        num_workers=num_workers,
        drop_last=False
    )

    return tuple(DataLoader(dataset, **loader_args) for dataset in datasets.values())

In [4]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

def validate_garchnet_dataset(data_path, window_size=10, dataset_type='kdd17'):
    """
    Validate the GARCHNet dataset implementation

    Args:
        data_path: Path to data directory
        window_size: Size of sliding window
        dataset_type: Type of dataset ('kdd17' or 'acl18')
    """
    dataset = GARCHNetDataset(data_path, window_size, dataset_type, split='train')
    ds_config = dataset.ds_config

    print("Dataset Configuration:")
    print(f"Dataset type: {dataset_type}")
    print(f"Window size: {window_size}")
    print(f"Train date cutoff: {ds_config['train_date']}")
    print(f"Validation date cutoff: {ds_config['val_date']}")
    print(f"Test date cutoff: {ds_config['test_date']}")

    # Load date index
    date_path = Path(data_path) / ds_config['date_path']
    print("\nLoading date index...")
    index2date = pd.read_csv(date_path, header=None).to_dict()[0]
    print(f"Total dates in index: {len(index2date)}")

    # Load and check tick files
    print("\nChecking ticker files...")
    data_dir = Path(data_path) / ds_config['path']
    tick_files = [p for p in data_dir.glob('*') if not p.is_dir()]
    print(f"Total ticker files found: {len(tick_files)}")

    # Validate first ticker's data
    print("\nValidating first ticker's data processing...")
    first_ticker_data = dataset.processed_data[0]
    print(f"Number of features: {len(first_ticker_data.columns) - 1}")  # -1 for date column
    print("\nFeatures:")
    for col in first_ticker_data.columns:
        if col != 'date':
            print(f"- {col}")

    # Check data splits
    print("\nChecking data splits...")
    train_idx, val_idx, test_idx = dataset.split_data(first_ticker_data)
    print(f"Train samples: {train_idx.sum()}")
    print(f"Validation samples: {val_idx.sum()}")
    print(f"Test samples: {test_idx.sum()}")

    # Check sequence creation
    print("\nChecking sequence creation...")
    sequences, targets = dataset.create_sequences(first_ticker_data)
    print(f"Sequence shape: {sequences.shape}")
    print(f"Target shape: {targets.shape}")

    # Validate DataLoaders
    print("\nValidating DataLoaders...")
    train_loader, val_loader, test_loader = create_dataloaders(
        data_path,
        window_size,
        batch_size=32,
        dataset_type=dataset_type
    )

    loaders = {
        'Train': train_loader,
        'Validation': val_loader,
        'Test': test_loader
    }

    loader_stats = {}
    for name, loader in loaders.items():
        print(f"\nChecking {name} loader:")
        batch_sequences, batch_targets = next(iter(loader))
        print(f"Batch sequence shape: {batch_sequences.shape}")
        print(f"Batch target shape: {batch_targets.shape}")

        # Check for NaN values
        print(f"NaN in sequences: {torch.isnan(batch_sequences).any().item()}")
        print(f"NaN in targets: {torch.isnan(batch_targets).any().item()}")

        # Value ranges
        print(f"Sequence min: {batch_sequences.min().item():.4f}")
        print(f"Sequence max: {batch_sequences.max().item():.4f}")
        print(f"Target min: {batch_targets.min().item():.4f}")
        print(f"Target max: {batch_targets.max().item():.4f}")

        loader_stats[name.lower()] = {
            'batch_sequences': batch_sequences,
            'batch_targets': batch_targets,
            'sequence_shape': batch_sequences.shape,
            'target_shape': batch_targets.shape
        }

    return {
        'dataset': dataset,
        'sequences': sequences,
        'targets': targets,
        'loader_stats': loader_stats,
        'data_splits': {
            'train': train_idx.sum(),
            'validation': val_idx.sum(),
            'test': test_idx.sum()
        }
    }

if __name__ == "__main__":
    # Set your data path
    data_path = "/content/drive/MyDrive/Colab Notebooks/google-colab/data"
    window_size = 10

    # Run validation
    validation_results = validate_garchnet_dataset(data_path, window_size)

    # Print summary
    print("\nValidation Summary:")
    print(f"Dataset size: {len(validation_results['dataset'])}")
    print("\nData split sizes:")
    for split, size in validation_results['data_splits'].items():
        print(f"- {split}: {size}")
    print(f"\nSequence window size: {window_size}")

    print("\nLoader shapes:")
    for split, stats in validation_results['loader_stats'].items():
        print(f"- {split}:")
        print(f"  Sequence shape: {stats['sequence_shape']}")
        print(f"  Target shape: {stats['target_shape']}")

Dataset Configuration:
Dataset type: kdd17
Window size: 10
Train date cutoff: 2015-01-01
Validation date cutoff: 2016-01-01
Test date cutoff: 2017-01-01

Loading date index...
Total dates in index: 2518

Checking ticker files...
Total ticker files found: 50

Validating first ticker's data processing...
Number of features: 19

Features:
- open
- high
- low
- close
- adj_close
- volume_change
- log_volume
- volume_ma5
- volume_std5
- volume_ma10
- volume_std10
- volume_ma20
- volume_std20
- price_trend5
- price_trend10
- price_trend15
- price_trend20
- price_trend25
- price_trend30

Checking data splits...
Train samples: 1985
Validation samples: 252
Test samples: 252

Checking sequence creation...
Sequence shape: (2479, 10, 19)
Target shape: (2479, 19)

Validating DataLoaders...
Train sequences: 98750
Val sequences: 12100
Test sequences: 12100

Checking Train loader:
Batch sequence shape: torch.Size([32, 10, 19])
Batch target shape: torch.Size([32, 19])
NaN in sequences: False
NaN in tar

In [5]:
import torch
import torch.nn as nn

class GARCHNetLSTM(nn.Module):
    def __init__(self, input_size, num_stocks, hidden_size=64, num_lstm_layers=2):
        super().__init__()

        # Create individual attention-LSTM for each stock
        self.stock_attention_lstm = nn.ModuleList([
            PerStockAttentionLSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_lstm_layers)
            for _ in range(num_stocks)
        ])

        # 3-layer MLP with GELU activation
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, hidden_size),
            nn.GELU()
        )

        # Output layers for market portfolio parameters
        self.variance_layer = nn.Linear(hidden_size, 1)
        self.dof_layer = nn.Linear(hidden_size, 1)
        self.skewness_layer = nn.Linear(hidden_size, 1)

        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        """
        Forward pass

        Args:
            x: Input tensor of shape (batch_size, num_stocks, window_size, num_features)

        Returns:
            tuple: (variance, dof, skewness)
        """
        # Process each stock through its attention-LSTM
        stock_contexts = []
        for i, stock_lstm in enumerate(self.stock_attention_lstm):
            stock_context = stock_lstm(x[:, i, :, :])  # x[:, i]: (batch_size, window_size, num_features)
            stock_contexts.append(stock_context)

        # Combine stock contexts (mean pooling for simplicity)
        combined_context = torch.mean(torch.stack(stock_contexts, dim=1), dim=1)  # Shape: (batch_size, hidden_size)

        # Pass through 3-layer MLP with GELU activation
        mlp_out = self.mlp(combined_context)  # Shape: (batch_size, hidden_size)

        # Compute market parameters
        variance = torch.log1p(torch.exp(self.variance_layer(mlp_out))).squeeze(-1)
        dof = torch.log1p(torch.exp(self.dof_layer(mlp_out))).squeeze(-1) + 2
        skewness = torch.tanh(self.skewness_layer(mlp_out)).squeeze(-1)

        return variance, dof, skewness


In [6]:
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from pathlib import Path
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

class HansenSkewedTNLLLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def compute_ct(self, nu):
        """Compute ct coefficient for Hansen's skewed t-distribution"""
        return torch.exp(
            torch.lgamma((nu + 1) / 2) -
            torch.lgamma(nu / 2) -
            0.5 * torch.log(torch.pi * (nu - 2))
        )

    def forward(self, returns_sequence, variance_sequence, dof_sequence, skewness_sequence):
        """
        Compute negative log-likelihood for Hansen's skewed t-distribution over the entire sequence

        Args:
            returns_sequence: Observed returns time series (batch_size, sequence_length)
            variance_sequence: Predicted variance series (batch_size, sequence_length)
            dof_sequence: Degrees of freedom series (batch_size, sequence_length)
            skewness_sequence: Skewness parameter series (batch_size, sequence_length)
        """
        # Compute standard deviation sequence
        sigma_sequence = torch.sqrt(variance_sequence)

        # Compute standardized residuals sequence (zero mean assumption)
        z_sequence = returns_sequence / sigma_sequence

        # Compute ct coefficient sequence
        ct_sequence = self.compute_ct(dof_sequence)

        # Compute squared terms sequence
        squared_term_sequence = returns_sequence**2 / ((dof_sequence - 2) * variance_sequence)

        # Compute sign terms sequence
        sign_term_sequence = torch.sign(returns_sequence)

        # Compute log-likelihood for each timestep in the sequence
        log_likelihood_sequence = (
            torch.log(torch.tensor(2.0)) -
            0.5 * torch.log(variance_sequence) +
            torch.log(ct_sequence) +
            (-(dof_sequence + 1) / 2) * torch.log1p(squared_term_sequence) +
            torch.log(torch.abs(sign_term_sequence))
        )

        # Sum log-likelihoods across time dimension for each batch
        batch_log_likelihood = torch.sum(log_likelihood_sequence, dim=1)

        # Return negative mean across batch
        return -torch.mean(batch_log_likelihood)


In [7]:
def train_garchnet(model, train_loader, val_loader, num_epochs, learning_rate=0.001, device='cuda' if torch.cuda.is_available() else 'cpu'):
    """Train GARCHNet model"""
    print(f"Using device: {device}")
    model = model.to(device)

    criterion = HansenSkewedTNLLLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = ReduceLROnPlateau(optimizer, 'min', patience=5)

    best_val_loss = float('inf')
    train_losses = []
    val_losses = []

    def get_adj_close_returns(batch_data):
        """Extract adjusted close returns from batch data"""
        # Find the index of adj_close in the features
        adj_close_idx = 4  # Assuming adj_close is at index 4 based on feature order
        return batch_data[..., adj_close_idx]

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0
        num_batches = 0

        for sequences, targets in train_loader:
            sequences = sequences.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()

            # Get adjusted close returns sequence
            returns_sequence = get_adj_close_returns(sequences)

            # Forward pass - get parameter sequences
            variance_sequence, dof_sequence, skewness_sequence = model(sequences)

            # Compute loss using the proper NLL
            loss = criterion(
                returns_sequence,
                variance_sequence,
                dof_sequence,
                skewness_sequence
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()
            num_batches += 1

        avg_train_loss = train_loss / num_batches
        train_losses.append(avg_train_loss)

        # Validation phase
        model.eval()
        val_loss = 0
        num_val_batches = 0

        with torch.no_grad():
            for sequences, targets in val_loader:
                sequences = sequences.to(device)
                targets = targets.to(device)

                # Get adjusted close returns sequence
                returns_sequence = get_adj_close_returns(sequences)

                variance_sequence, dof_sequence, skewness_sequence = model(sequences)

                loss = criterion(
                    returns_sequence,
                    variance_sequence,
                    dof_sequence,
                    skewness_sequence
                )
                val_loss += loss.item()
                num_val_batches += 1

        avg_val_loss = val_loss / num_val_batches
        val_losses.append(avg_val_loss)
        scheduler.step(avg_val_loss)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': avg_train_loss,
                'val_loss': avg_val_loss,
            }, 'best_garchnet_model.pth')

        print(f'Epoch [{epoch+1}/{num_epochs}]')
        print(f'Training Loss: {avg_train_loss:.6f}')
        print(f'Validation Loss: {avg_val_loss:.6f}')
        print(f'Learning Rate: {optimizer.param_groups[0]["lr"]:.6f}\n')

    # Plot training history
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Negative Log-Likelihood')
    plt.title('GARCHNet Training History')
    plt.legend()
    plt.grid(True)
    plt.show()

    return train_losses, val_losses

def main():
    # Set random seeds for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)

    # Training parameters
    data_path = "/content/drive/MyDrive/Colab Notebooks/google-colab/data"
    window_size = 10
    batch_size = 32
    hidden_size = 64
    num_epochs = 100
    learning_rate = 0.001

    # Create data loaders
    train_loader, val_loader, test_loader = create_dataloaders(
        data_path=data_path,
        window_size=window_size,
        batch_size=batch_size,
        dataset_type='kdd17'
    )

    # Get input size from a sample batch
    sample_sequences, _ = next(iter(train_loader))
    input_size = sample_sequences.shape[-1]

    # Initialize model
    model = GARCHNetLSTM(
        input_size=input_size,
        hidden_size=hidden_size
    )

    # Train model
    train_losses, val_losses = train_garchnet(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        num_epochs=num_epochs,
        learning_rate=learning_rate
    )

    print("Training complete!")
    return model, train_losses, val_losses

if __name__ == "__main__":
    main()

Train sequences: 98750
Val sequences: 12100
Test sequences: 12100


TypeError: GARCHNetLSTM.__init__() missing 1 required positional argument: 'num_stocks'